# Hazard Event Ingestion

Builds the historical hazard-event table (2015-2025) from the raw SNGRE/ECU911 workbook: filters the hazards in scope, validates dates and coordinates, removes personal data, builds the complete monthly ENSO series and joins both.

Outputs: `hazard_event` and `enso_index_monthly` in `data/processed/`.

## 0. Setup

In [ ]:

# Standard library: system path handling
import sys
# Object-oriented file paths
from pathlib import Path

# Project root is the parent of the notebooks folder
PROJECT_ROOT = Path.cwd().parent
# Make the project's src package importable
sys.path.insert(0, str(PROJECT_ROOT))

# Helpers to load the YAML config and resolve folder paths
from src.utilidades.config_utils import load_config, resolve_path
# Project logger factory
from src.utilidades.logging_utils import get_logger
# Helpers to read an Excel sheet and write processed tables
from src.utilidades.io_utils import read_excel_sheet, write_processed_table

# Tabular data handling
import pandas as pd
# Numerical operations
import numpy as np
# Regular expressions
import re

# Show up to 30 columns when printing DataFrames
pd.set_option("display.max_columns", 30)

In [ ]:
# Path to the central configuration file
CONFIG_PATH = PROJECT_ROOT / "config" / "config.yaml"
# Load the configuration into a dictionary
config = load_config(CONFIG_PATH)

# Create a logger that writes to the logs folder
logger = get_logger("step1_ingestion", config, log_dir=PROJECT_ROOT / "logs")
# Log that the notebook started and which config was used
logger.info("ingesta iniciada — configuración cargada desde %s", CONFIG_PATH)

## 1. Load the raw event table

In [ ]:
# Path to the raw source workbook
workbook_path = resolve_path(config, "raw_workbook", base_dir=Path.cwd())

# Read the raw hazard-events sheet
raw_events = read_excel_sheet(workbook_path, config["sheets"]["hazard_events"])

# Log the number of rows and columns loaded
logger.info("Base cruda cargada: %d filas x %d columnas", *raw_events.shape)

# Preview the first three rows
raw_events.head(3)

## 1.1 Auditable data corrections

In [ ]:
# Known data corrections declared in the config
corrections = config.get("known_data_corrections", [])
# Apply each correction one by one
for corr in corrections:
    # Select the event to correct by its ID
    mask = raw_events["ID"] == corr["event_id"]
    # Count matching rows
    n_afectados = int(mask.sum())
    # Each correction must match exactly one event
    assert n_afectados == 1, (
        f"se esperaba exactamente 1 fila con ID={corr['event_id']}, "
        f"se encontraron {n_afectados} — revisar antes de aplicar la corrección"
    )
    # Read the current value
    valor_anterior = raw_events.loc[mask, corr["column"]].iloc[0]
    # Check the current value is the one the correction expects
    assert valor_anterior == corr["original_value"], (
        f"el valor original en el workbook ({valor_anterior}) ya no coincide con "
        f"'original_value' en config.yaml ({corr['original_value']}) — "
        "¿cambió el workbook fuente? Revisar antes de aplicar la corrección."
    )
    # Write the corrected value
    raw_events.loc[mask, corr["column"]] = corr["corrected_value"]
    # Log a warning so the correction is auditable
    logger.warning(
        "Corrección de dato aplicada: evento ID=%s, columna '%s': %s -> %s "
        "(decidido por %s, %s)",
        corr["event_id"], corr["column"], valor_anterior, corr["corrected_value"],
        corr["decided_by"], corr["decided_date"],
    )

# Log how many corrections were applied
logger.info("%d corrección(es) de datos aplicadas sobre raw_events", len(corrections))

## 2. Filter by hazard scope

In [ ]:
# Mapping of raw event names to hazard types in scope
hazard_scope = config["hazard_scope"]

# Keep only events inside the hazard scope
events = raw_events[raw_events["EVENTO"].isin(hazard_scope)].copy()

# Add a standardized hazard type column
events["hazard_type"] = events["EVENTO"].map(hazard_scope)

# Log how many events are in scope
logger.info("Eventos dentro del alcance de peligro: %d de %d", len(events), len(raw_events))
# Show the number of events per hazard type
events["hazard_type"].value_counts()

## 3. Flood mechanism sub-type

In [ ]:
# Keywords that identify each flood sub-type, from the config
subtype_keywords = config["flood_subtype_keywords"]

# Classify the flooding mechanism from the free-text cause
def classify_flood_subtype(causa: str) -> str:
    """Classify the flooding mechanism (for example river overflow or flash flood) from the free-text cause.
    
    Args:
        causa: Free-text cause reported for the event.
    
    Returns:
        The sub-type label, 'unknown' for non-text values, or 'other' when no keyword matches.
    """
    # Non-text causes cannot be classified
    if not isinstance(causa, str):
        # Return 'unknown' for missing causes
        return "unknown"
    # Lower-case the text to match keywords
    causa_lower = causa.lower()
    # Check each sub-type in order
    for subtype, keywords in subtype_keywords.items():
        # A sub-type matches if any of its keywords appears in the text
        if any(kw in causa_lower for kw in keywords):
            # Return the first matching sub-type
            return subtype
    # Default label when no keyword matches
    return "other"

# Flag flood-like events (floods and debris flows)
is_flood_like = events["hazard_type"].isin(["flood", "debris_flow"])
# Assign the sub-type only to flood-like events
events["flood_subtype"] = np.where(
    is_flood_like,
    events["CAUSA"].apply(classify_flood_subtype),
    None,
)

# Log that the sub-type distribution was computed
logger.info("Distribución de flood_subtype calculada")
# Show the sub-type distribution
events.loc[is_flood_like, "flood_subtype"].value_counts()

## 4. Dates and project time window (2015-2025)

In [ ]:
# Parse event dates, turning invalid dates into missing values
events["event_date"] = pd.to_datetime(events["FECHA DEL EVENTO"], errors="coerce")

# Extract the year
events["year"] = events["event_date"].dt.year
# Extract the month
events["month"] = events["event_date"].dt.month

# Count dates that could not be parsed
n_unparsed_dates = events["event_date"].isna().sum()
# Log the number of unparseable dates
logger.info("Fechas no parseables: %d de %d filas", n_unparsed_dates, len(events))

# First year of the analysis window
year_min = config["date_range"]["start_year"]
# Last year of the analysis window
year_max = config["date_range"]["end_year"]
# Flag events inside the analysis window
in_range = events["year"].between(year_min, year_max)
# Count events outside the window
n_out_of_range = (~in_range).sum()
# Log the number of events outside the window
logger.info("Filas fuera del rango %d-%d: %d", year_min, year_max, n_out_of_range)

# Keep only events inside the window
events = events[in_range].copy()
# Log the number of rows after the temporal filter
logger.info("Filas tras recorte temporal: %d", len(events))

## 5. Coordinate validation

In [ ]:
# Bounding box of Ecuador from the config
bbox = config["ecuador_bbox"]

# Check latitude is inside the bounding box
lat_ok = events["LATITUD"].between(bbox["lat_min"], bbox["lat_max"])
# Check longitude is inside the bounding box
lon_ok = events["LONGITUD"].between(bbox["lon_min"], bbox["lon_max"])
# A coordinate is valid when both latitude and longitude are inside
events["coord_valid"] = lat_ok & lon_ok

# Count missing coordinates
n_null_coords = events["LATITUD"].isna().sum() + events["LONGITUD"].isna().sum()
# Count coordinates outside the bounding box
n_invalid_coords = (~events["coord_valid"]).sum()
# Log both counts
logger.info("Coordenadas nulas: %d | Coordenadas fuera de bbox Ecuador: %d", n_null_coords, n_invalid_coords)

# Show how many events have valid coordinates
events["coord_valid"].value_counts()

## 6. Personal data (PII) handling

In [ ]:
# Regular expression that detects phone numbers
phone_pattern = config["pii"]["phone_regex"]

# Flag events whose source text may contain personal data
events["source_has_pii"] = events["Fuente"].astype(str).str.contains(
    phone_pattern, regex=True, na=False
)

# Percentage of events flagged
pct_pii = events["source_has_pii"].mean() * 100
# Log the percentage
logger.info("Filas con posible PII en 'Fuente': %.1f%%", pct_pii)

# Record the source type of every event
events["source_type"] = "SNGRE_ECU911_report"

# Drop the free-text source column if the config says so
if config["pii"]["drop_raw_source_text"]:
    # Remove the column that may contain personal data
    events = events.drop(columns=["Fuente"])
    # Log that the column was removed
    logger.info("Columna 'Fuente' (texto libre con PII) eliminada de la tabla procesada")

## 7. Duplicate check

In [ ]:
# Count duplicated event IDs
n_duplicated_ids = events["ID"].duplicated().sum()
# Log the number of duplicates
logger.info("IDs duplicados dentro del alcance filtrado: %d", n_duplicated_ids)

# Warn when duplicates exist
if n_duplicated_ids > 0:
    # Warning asking for review before continuing
    logger.warning("Se detectaron IDs duplicados — revisar antes de continuar")

## 8. Complete monthly ENSO series

In [ ]:
# Mapping from running quarters (e.g. DJF) to a month
trimester_to_month = config["trimester_to_month"]

# Read the quarterly ICEN sheet
icen_trimestral = read_excel_sheet(workbook_path, config["sheets"]["icen_trimestral"])
# Read the quarterly ONI sheet
oni_trimestral = read_excel_sheet(workbook_path, config["sheets"]["oni_trimestral"])

# Drop ICEN rows without a year
icen_trimestral = icen_trimestral.dropna(subset=["Año"])
# Drop ONI rows without a year
oni_trimestral = oni_trimestral.dropna(subset=["Año"])

# Convert ICEN quarters to months
icen_trimestral["month"] = icen_trimestral["Trimestre"].map(trimester_to_month)
# Convert ONI quarters to months
oni_trimestral["month"] = oni_trimestral["Trimestre"].map(trimester_to_month)

# Integer year for ICEN
icen_trimestral["year"] = icen_trimestral["Año"].astype(int)
# Integer year for ONI
oni_trimestral["year"] = oni_trimestral["Año"].astype(int)

In [ ]:
# Start the monthly series from ICEN with standardized column names
enso_monthly = icen_trimestral[["year", "month", "ICEN", "Categoría y magnitud"]].rename(
    columns={"Categoría y magnitud": "icen_category"}
).merge(
    oni_trimestral[["year", "month", "ONI", "Categoría y magnitud"]].rename(
        columns={"Categoría y magnitud": "oni_category"}
    ),
    on=["year", "month"],
    how="outer",
)

# Keep only the months inside the analysis window
enso_monthly = enso_monthly[
    (enso_monthly["year"] >= year_min) & (enso_monthly["year"] <= year_max)
].sort_values(["year", "month"]).reset_index(drop=True)

# Expected number of months: 12 per year
expected_rows = (year_max - year_min + 1) * 12
# Log the length of the monthly series
logger.info(
    "ENSO_Index_Monthly: %d filas (esperado %d) | nulos ICEN=%d ONI=%d",
    len(enso_monthly), expected_rows,
    enso_monthly["ICEN"].isna().sum(), enso_monthly["ONI"].isna().sum(),
)
# Check the monthly series is complete
assert len(enso_monthly) == expected_rows, "Serie ENSO mensual incompleta — revisar fuente"

# Preview the first 12 months
enso_monthly.head(12)

## 9. Join events to the monthly ENSO series

In [ ]:
# Join each event to the ENSO state of its month
events = events.merge(enso_monthly, on=["year", "month"], how="left")

# Share of events that received an ENSO value
match_rate = events["ICEN"].notna().mean()
# Log the match rate
logger.info("Tasa de emparejamiento evento-ENSO: %.1f%%", match_rate * 100)
# Check every event matched an ENSO month
assert match_rate == 1.0, "Hay eventos sin ENSO asociado — revisar rango de fechas"

## 10. Final column selection and renaming

In [ ]:
# Mapping from raw column names to the final schema
column_mapping = {
    "ID": "event_id",
    "PROVINCIA": "province",
    "CODIFICACION PROVINCIAL": "province_code",
    "CANTON": "canton",
    "CODIFICACION CANTONAL": "canton_code",
    "PARROQUIAS": "parish",
    "CODIFICACION PARROQUIAL": "parish_code",
    "LATITUD": "latitude",
    "LONGITUD": "longitude",
    "CAUSA": "cause_raw",
    "FALLECIDAS": "deaths",
    "HERIDAS": "injured",
    "PERSONAS DESAPARECIDAS": "missing_persons",
    "PERSONAS AFECTADAS DIRECTAMENTE": "affected_direct",
    "AFECTADAS INDIRECTAS": "affected_indirect",
    "PERSONAS DAMNIFICADAS": "affected_severe",
    "PERSONAS EVACUADAS": "evacuated",
    "PERSONAS ALBERGADAS": "sheltered",
    "VIVIENDAS AFECTADAS": "houses_affected",
    "VIVIENDAS DESTRUIDAS": "houses_destroyed",
    "ESTABLECIMIENTOS EDUCATIVOS AFECTADOS": "schools_affected",
    "ESTABLECIMIENTOS EDUCATIVOS DESTRUIDOS": "schools_destroyed",
    "CENTROS DE SALUD AFECTADOS": "health_facilities_affected",
    "CENTROS DE SALUD DESTRUIDOS": "health_facilities_destroyed",
    "Ha. CULTIVO AFECTADAS": "crops_affected_ha",
    "Ha. CULTIVO PERDIDAS": "crops_destroyed_ha",
    "ANIMALES AFECTADOS": "livestock_affected",
    "ANIMALES MUERTOS": "livestock_deaths",
}

# Columns created in this notebook
own_columns = [
    "event_date", "year", "month", "hazard_type", "flood_subtype",
    "coord_valid", "source_has_pii", "source_type",
    "ICEN", "icen_category", "ONI", "oni_category",
]

# Raw columns to keep
raw_cols_to_keep = list(column_mapping.keys())

# Select and rename columns to the final schema
hazard_event = events[raw_cols_to_keep + own_columns].rename(columns=column_mapping)

# Log the size of the final table
logger.info("Tabla Hazard_Event final: %d filas x %d columnas", *hazard_event.shape)
# Preview the first three rows
hazard_event.head(3)

## 11. Data-quality summary

In [ ]:
# Dictionary with the data-quality summary
summary = {
    "total_eventos_alcance": len(hazard_event),
    "rango_fechas": (str(hazard_event["event_date"].min()), str(hazard_event["event_date"].max())),
    "por_hazard_type": hazard_event["hazard_type"].value_counts().to_dict(),
    "coordenadas_invalidas_pct": round((~hazard_event["coord_valid"]).mean() * 100, 2),
    "provincias_unicas": hazard_event["province"].nunique(),
    "cantones_unicos": hazard_event["canton_code"].nunique(),
    "tasa_emparejamiento_enso_pct": 100.0,
    "filas_con_pii_removida_pct": round(hazard_event["source_has_pii"].mean() * 100, 2),
}

# Print each summary item
for key, value in summary.items():
    # Print the key and its value
    print(f"{key}: {value}")

# Log the quality summary
logger.info("Resumen de calidad de datos generado: %s", summary)

## 12. Export

In [ ]:
# Folder for processed tables
processed_dir = resolve_path(config, "processed_dir", base_dir=Path.cwd())

# Write the hazard events table
hazard_event_paths = write_processed_table(hazard_event, processed_dir, "hazard_event")

# Write the monthly ENSO series
enso_monthly_paths = write_processed_table(enso_monthly, processed_dir, "enso_index_monthly")

# Log the exported files
logger.info("Exportación completa: %s", {**hazard_event_paths, **enso_monthly_paths})
# Print the export folder
print("Archivos escritos en:", processed_dir)